In [6]:
%pip install -q langchain-core langchain-openai pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.


# Task 1, Structured Profile Generator

In [7]:
from pydantic import BaseModel, Field
from typing import List


class UserProfile(BaseModel):
    name: str = Field(description="The person's name")
    age: int = Field(description="The person's age in years")
    profession: str = Field(description="The person's profession")
    skills: List[str] = Field(description="A list of professional skills")
    career_goal: str = Field(description="The person's main career goal")

In [8]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
base_url = os.getenv("OPENROUTER_BASE_URL")
model_name = os.getenv("MODEL_NAME")

if not api_key:
    raise ValueError(
        "OPENROUTER_API_KEY was not found."
    )

parser = PydanticOutputParser(pydantic_object=UserProfile)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a profile generation assistant. "
        "Extract the requested profile information carefully. "
        "If a detail is missing, use a reasonable value only when "
        "it can be inferred; otherwise use an explicit placeholder "
        "consistent with the required field type.\n"
        "{format_instructions}"
    ),
    (
        "human",
        "Create a structured profile from this request:\n{request}"
    )
]).partial(
    format_instructions=parser.get_format_instructions()
)

llm = ChatOpenAI(
    model=model_name,
    base_url=base_url,
    api_key=api_key,
    temperature=0
)

profile_chain = prompt | llm | parser

print("Profile generation chain created successfully.")

Profile generation chain created successfully.


In [9]:
requests = [
    (
        "My name is Ahmed, I am 25 years old, "
        "a Python developer skilled in Django, SQL, and Git. "
        "My goal is to become a backend engineer."
    ),
    (
        "Create a profile for Sara, a 23-year-old computer "
        "engineering graduate who knows Python, machine learning, "
        "and PyTorch. She wants to become an AI engineer."
    )
]

results = []

for i, user_request in enumerate(requests, start=1):
    print(f"\n{'=' * 60}")
    print(f"Input {i}:")
    print(user_request)

    profile = profile_chain.invoke({"request": user_request})
    results.append(profile)

    print("\nParsed Profile:")
    print(profile.model_dump_json(indent=2))

    print("\nSelected Field:")
    print("Name:", profile.name)


Input 1:
My name is Ahmed, I am 25 years old, a Python developer skilled in Django, SQL, and Git. My goal is to become a backend engineer.



Parsed Profile:
{
  "name": "Ahmed",
  "age": 25,
  "profession": "Python developer",
  "skills": [
    "Django",
    "SQL",
    "Git"
  ],
  "career_goal": "become a backend engineer"
}

Selected Field:
Name: Ahmed

Input 2:
Create a profile for Sara, a 23-year-old computer engineering graduate who knows Python, machine learning, and PyTorch. She wants to become an AI engineer.

Parsed Profile:
{
  "name": "Sara",
  "age": 23,
  "profession": "Computer Engineering Graduate",
  "skills": [
    "Python",
    "machine learning",
    "PyTorch"
  ],
  "career_goal": "AI engineer"
}

Selected Field:
Name: Sara


In [10]:
response = (prompt | llm).invoke({"request": requests[0]})

print("Raw response:")
print(repr(response.content))

Raw response:
'{\n  "name": "Ahmed",\n  "age": 25,\n  "profession": "Python developer",\n  "skills": [\n    "Django",\n    "SQL",\n    "Git"\n  ],\n  "career_goal": "Become a backend engineer"\n}'


*************************

# Task 2, Streaming and Batch Concept Explainer

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Tell me the capital of {country} in one sentence."
)

parser = StrOutputParser()

capital_chain = prompt | llm | parser

In [12]:
country = "Japan"

print(f"Capital of {country}:")

for chunk in capital_chain.stream({"country": country}):
    print(chunk, end="", flush=True)

print()

Capital of Japan:
The capital of Japan is Tokyo.


In [13]:
countries = ["Egypt", "France", "Brazil"]

inputs = [{"country": country} for country in countries]

results = capital_chain.batch(inputs)

for country, result in zip(countries, results):
    print(f"{country}: {result}")

Egypt: The capital of Egypt is Cairo.
France: The capital of France is Paris.
Brazil: User Safety: safe


*******************************************

# Task 3, Mini RAG Q&A App

In [14]:
%pip install -q langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers
%pip install -U lazy_loader librosa

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [15]:
text = """
Solar energy is a renewable source of energy that comes from the sun.
Solar panels convert sunlight into electricity using photovoltaic cells.
Solar energy can be used to power homes, schools, and businesses.
It can help reduce dependence on fossil fuels.
Solar energy produces electricity without direct carbon emissions during operation.
The amount of electricity generated depends on sunlight and weather conditions.
Solar batteries can store extra electricity for later use.
Regular cleaning and maintenance can help solar panels work efficiently.
"""

with open("solar_energy.txt", "w", encoding="utf-8") as file:
    file.write(text.strip())

print("Document created successfully.")

Document created successfully.


In [16]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("solar_energy.txt", encoding="utf-8")
documents = loader.load()

document_text = documents[0].page_content

print(document_text)

C:\Users\dell\AppData\Local\Temp\ipykernel_9248\3577377672.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Solar energy is a renewable source of energy that comes from the sun.
Solar panels convert sunlight into electricity using photovoltaic cells.
Solar energy can be used to power homes, schools, and businesses.
It can help reduce dependence on fossil fuels.
Solar energy produces electricity without direct carbon emissions during operation.
The amount of electricity generated depends on sunlight and weather conditions.
Solar batteries can store extra electricity for later use.
Regular cleaning and maintenance can help solar panels work efficiently.


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")

for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i}:")
    print(chunk.page_content)

Number of chunks: 3

Chunk 1:
Solar energy is a renewable source of energy that comes from the sun.
Solar panels convert sunlight into electricity using photovoltaic cells.
Solar energy can be used to power homes, schools, and businesses.

Chunk 2:
It can help reduce dependence on fossil fuels.
Solar energy produces electricity without direct carbon emissions during operation.
The amount of electricity generated depends on sunlight and weather conditions.

Chunk 3:
Solar batteries can store extra electricity for later use.
Regular cleaning and maintenance can help solar panels work efficiently.


In [18]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Embeddings created and stored in FAISS.")

c:\Users\dell\.conda\envs\route_assignments\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1620.12it/s]


Embeddings created and stored in FAISS.


In [19]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully.")

Retriever created successfully.


In [20]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the information in the context below.

If the answer is not available in the context, say:
"I don't know based on the provided document."

Do not use outside knowledge or invent information.

Context:
{context}

Question:
{question}

Answer:
""")

rag_chain = (
    {
        "context": itemgetter("question") | retriever | RunnableLambda(format_docs),
        "question": itemgetter("question")
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully.")

RAG chain created successfully.


In [21]:
questions = [
    "How do solar panels generate electricity?",
    "What are two benefits of solar energy?",
    "How much does installing solar panels cost in Egypt?"
]

for i, question in enumerate(questions, start=1):
    print(f"\nQuestion {i}: {question}")

    answer = rag_chain.invoke({"question": question})

    print(f"Answer: {answer}")


Question 1: How do solar panels generate electricity?
Answer: Solar panels generate electricity by converting sunlight into electricity using photovoltaic cells.

Question 2: What are two benefits of solar energy?
Answer: Based on the provided document, two benefits of solar energy are:
1. It can help reduce dependence on fossil fuels.
2. Solar energy produces electricity without direct carbon emissions during operation.

Question 3: How much does installing solar panels cost in Egypt?
Answer: I don't know based on the provided document.


**************************************

# Task 4, Retriever Comparison Lab

In [22]:
%pip install -q rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [23]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 8, "lambda_mult": 0.5}
)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, similarity_retriever],
    weights=[0.5, 0.5]
)

print("All three retrievers are ready.")

All three retrievers are ready.


In [24]:
questions = [
    "How do solar panels generate electricity?",
    "What are two benefits of solar energy?",
    "How much does installing solar panels cost in Egypt?"
]

retrievers = {
    "Similarity": similarity_retriever,
    "MMR": mmr_retriever,
    "Ensemble (BM25 + Similarity)": ensemble_retriever
}

for question in questions:
    print("\n" + "=" * 70)
    print(f"Question: {question}")

    for name, retriever in retrievers.items():
        print(f"\n{name}:")
        docs = retriever.invoke(question)

        for i, doc in enumerate(docs, start=1):
            print(f"{i}. {doc.page_content}")


Question: How do solar panels generate electricity?

Similarity:
1. Solar energy is a renewable source of energy that comes from the sun.
Solar panels convert sunlight into electricity using photovoltaic cells.
Solar energy can be used to power homes, schools, and businesses.
2. It can help reduce dependence on fossil fuels.
Solar energy produces electricity without direct carbon emissions during operation.
The amount of electricity generated depends on sunlight and weather conditions.
3. Solar batteries can store extra electricity for later use.
Regular cleaning and maintenance can help solar panels work efficiently.

MMR:
1. Solar energy is a renewable source of energy that comes from the sun.
Solar panels convert sunlight into electricity using photovoltaic cells.
Solar energy can be used to power homes, schools, and businesses.
2. Solar batteries can store extra electricity for later use.
Regular cleaning and maintenance can help solar panels work efficiently.
3. It can help reduc

************

# Task 5, Tool-Using Assistant

In [25]:
from datetime import datetime
from langchain_core.tools import tool


@tool
def get_current_time() -> str:
    """Get the current local time."""
    return datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %Z")


@tool
def get_number_of_words(sentence: str) -> int:
    """Count words in a sentence using spaces as separators."""
    return len(sentence.split(" "))


@tool
def search_knowledge_base(query: str) -> str:
    """Search the solar energy knowledge base for relevant information."""
    docs = retriever.invoke(query)

    if not docs:
        return "No relevant information was found."

    return "\n\n".join(doc.page_content for doc in docs)

In [26]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

try:
    from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
except ImportError:
    from langchain.agents import create_tool_calling_agent, AgentExecutor


tools = [
    get_current_time,
    get_number_of_words,
    search_knowledge_base
]

agent_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. Use the available tools whenever "
        "they are needed to answer the user's request. Do not guess "
        "results that can be obtained from a tool."
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent = create_tool_calling_agent(
    llm,
    tools,
    agent_prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,
    return_intermediate_steps=True
)

print("Agent is ready.")

Agent is ready.


In [27]:
test_prompts = [
    "What time is it right now?",
    "How many words are in this sentence: 'the quick brown fox jumps over the lazy dog'?",
    "What does the knowledge base say about how solar panels generate electricity?"
]

for question in test_prompts:
    print("\n" + "=" * 65)
    print("User:", question)

    result = agent_executor.invoke({"input": question})

    print("\nTools called:")
    if result["intermediate_steps"]:
        for action, observation in result["intermediate_steps"]:
            print(f"- {action.tool}")
            print(f"  Tool result: {observation}")
    else:
        print("No tools were called.")

    print("\nFinal answer:")
    print(result["output"])


User: What time is it right now?

Tools called:
- get_current_time
  Tool result: 2026-10-13 23:21:27 Egypt Daylight Time

Final answer:
The current time is **23:21:27 on October 13, 2026** (Egypt Daylight Time).

User: How many words are in this sentence: 'the quick brown fox jumps over the lazy dog'?

Tools called:
- get_number_of_words
  Tool result: 9

Final answer:
The sentence 'the quick brown fox jumps over the lazy dog' contains 9 words.

User: What does the knowledge base say about how solar panels generate electricity?

Tools called:
- search_knowledge_base
  Tool result: Solar energy is a renewable source of energy that comes from the sun.
Solar panels convert sunlight into electricity using photovoltaic cells.
Solar energy can be used to power homes, schools, and businesses.

Solar batteries can store extra electricity for later use.
Regular cleaning and maintenance can help solar panels work efficiently.

It can help reduce dependence on fossil fuels.
Solar energy produce

In [28]:
sentence = "the quick brown fox jumps over the lazy dog"

print("Sentence:", sentence)
print("Word count:", get_number_of_words.invoke({"sentence": sentence}))

Sentence: the quick brown fox jumps over the lazy dog
Word count: 9


****************************